<a href="https://colab.research.google.com/github/tyhardison/Research-Project-Do-LLMs-Play-Nash/blob/main/Final_Project_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Do LLMs Play Nash?**
# ***Final Project - Part 1***


---


**Name: Ty Hardison**

**Course: CSCE 361**

In [ ]:
# ── Mount Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Set your save directory — change the folder name as you like
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/LLM_Nash_Experiment/'

# Create the folder if it doesn't exist
import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Drive mounted. Saving to: {DRIVE_OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Saving to: /content/drive/MyDrive/LLM_Nash_Experiment/


In [ ]:
# ============================================================
#  Do LLMs Play Nash? — Data Collection Notebook
#  CSCE 631 — Algorithmic Game Theory meets LLMs
# ============================================================
# USAGE:
#   1. Paste your TAMU API key and CF_Authorization cookie below
#   2. Run all cells top to bottom
#   3. Results auto-save to llm_nash_results.csv after every trial
# ============================================================


# !pip install openai pandas numpy matplotlib seaborn --quiet
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import time
import json
import re
import os
import random
import uuid
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import pandas as pd
import numpy as np
from collections import Counter
from unittest.mock import MagicMock, patch

try:
    import openai
except ImportError:
    raise ImportError("Run: pip install openai")

# ── TAMU API Configuration ───────────────────────────────────
TAMU_API_KEY  = "(insert TAMU_API_KEY here)"   # your key
TAMU_BASE_URL = "https://chat.tamu.ai/api"
CF_COOKIE     = "CF_Authorization=(insert CF_Authorization code here)"

# ── Model Selection ──────────────────────────────────────────
# Change DEFAULT_MODEL and MAX_TOKENS_STANDARD together when switching models
#
# [Anthropic] Claude Haiku  (no thinking mode): MAX_TOKENS_STANDARD = 8
# [Anthropic] Claude Sonnet (thinking mode on): MAX_TOKENS_STANDARD = 16384

# [OpenAI] GPT-5.5 (no thinking on): MAX_TOKENS_STANDARD = 8
# [OpenAI] o4-mini (or 03) (thinking mode on): MAX_TOKENS_STANDARD = 16384

# [Google] Gemini-2.5-flash-light (no thinking on): MAX_TOKENS_STANDARD = 8
# [Google] Gemini-2.5-flash (thinking mode on): MAX_TOKENS_STANDARD = 16384


DEFAULT_MODEL        = "protected.gemini-2.5-flash"
MAX_TOKENS_STANDARD  = 16384       # ← change this when switching models
MAX_TOKENS_REASONING = 16384     # used only in Case 5 — safe for all models

# ── Experiment Settings ──────────────────────────────────────
TRIALS_PER_CONDITION = 30

OUTPUT_FILE          = DRIVE_OUTPUT_DIR + "llm_nash_results.csv" #remove "DRIVE_OUTPUT_DIR" to create file locally
REASONING_OUTPUT_FILE = DRIVE_OUTPUT_DIR + "llm_nash_reasoning.csv" #remove "DRIVE_OUTPUT_DIR" to create file locally
REASONING_TRIALS = 10

SLEEP_BETWEEN_CALLS  = 1.5     # seconds — be polite to the API
LIVE_MODE            = True    # set False to dry-run with mock responses


## **1) LLM Agent and Prompt Creation**


---



## 1.1) LLMAgent Class and Error Handling
This contains the implementation of agent call structure and how the API interaction will work

In [ ]:

# Creating the LLMAgent
class LLMAgent:
    """Wraps the TAMU API. Each call is fully independent — no shared state."""

    def __init__(self, model=DEFAULT_MODEL, system_prompt="You are a helpful assistant.",
                 temperature=1, cookie=CF_COOKIE, max_tokens=64):
        self.model        = model
        self.system_prompt = system_prompt
        self.temperature  = temperature
        self.max_tokens   = max_tokens
        self.total_tokens = 0

        self.client = openai.OpenAI(
            base_url=TAMU_BASE_URL,
            api_key=TAMU_API_KEY,
            default_headers={"Cookie": cookie},
        )

    def query(self, user_prompt: str) -> str:
        """Send a single isolated request. Returns response text."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user",   "content": user_prompt},
        ]
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=self.temperature,
                    max_tokens=self.max_tokens,
                    stream=True,
                    stream_options={"include_usage": True},
                )
                full_text = ""
                got_real_usage = False
                if isinstance(response, openai.Stream):
                    for chunk in response:
                        if not chunk.choices:
                            usage = getattr(chunk, "usage", None)
                            if usage:
                                self.total_tokens += getattr(usage, "total_tokens", 0)
                                got_real_usage = True
                            continue
                        delta = chunk.choices[0].delta
                        content = getattr(delta, "content", None)
                        if content:
                            full_text += content
                        usage = getattr(chunk, "usage", None)
                        if usage:
                            self.total_tokens += getattr(usage, "total_tokens", 0)
                            got_real_usage = True
                else:
                    full_text = response if isinstance(response, str) else response.choices[0].message.content

                if self._is_gateway_error(full_text):
                    raise RuntimeError(f"TAMU gateway error: {full_text[:500]}")

                if not got_real_usage:
                    self.total_tokens += (len(self.system_prompt) + len(user_prompt) + len(full_text)) // 4

                return full_text

            except openai.RateLimitError:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    raise
            except openai.AuthenticationError:
                raise RuntimeError("Cookie expired — refresh CF_COOKIE")

    def _is_gateway_error(self, text: str) -> bool:
        if not text:
            return False
        return any(m in text.lower() for m in ["budget exceeded", "unexpected error", "🚫"])

# 1.2) Prompt Library
Establishes the standardized selection of input prompts to be sent to the API for testing. Each Game is provided with 3 variations to account for prompt sensitivity in response outputs.

In [ ]:

# ── Game Definitions ─────────────────────────────────
# Each game has:
#   name        — display name
#   actions     — the labeled options shown to the model
#   nash        — the NE action label(s); list because some games have multiple
#   nash_type   — pure / mixed / multiple
#   nash_note   — human-readable NE description for the results table
#   prompts     — list of prompt variants (randomly sampled each trial)
#
# IMPORTANT: Every prompt variant ends with the same standardized instruction
# so responses are always parseable as a single letter.

STANDARD_SUFFIX = (
    "\n\nRespond with ONLY a single letter from the options above. "
    "Do not explain. Do not add punctuation. Just the letter."
)



GAMES = {

    "prisoners_dilemma": {
        "name": "Prisoner's Dilemma",
        "actions": {"A": "Cooperate (stay silent)", "B": "Defect (testify)"},
        "nash": ["B"],
        "nash_type": "pure",
        "nash_note": "Defect (B) — strictly dominant strategy",
        "prompts": [
            (
                "You and another suspect are held in separate rooms. You cannot communicate.\n"
                "If both stay silent: you each serve 1 year.\n"
                "If you testify and they stay silent: you go free, they serve 4 years.\n"
                "If they testify and you stay silent: they go free, you serve 4 years.\n"
                "If both testify: you each serve 3 years.\n\n"
                "Options:\n  A) Cooperate (stay silent)\n  B) Defect (testify)"
            ),
            (
                "Two partners in crime are interrogated separately. No communication is possible.\n"
                "Silence from both: 1 year each.\n"
                "You talk, they stay silent: you walk free, they get 4 years.\n"
                "They talk, you stay silent: they walk free, you get 4 years.\n"
                "Both talk: 3 years each.\n\n"
                "Options:\n  A) Stay silent\n  B) Talk to the police"
            ),
            (
                "You are a suspect in an interrogation. The other suspect is in a separate room.\n"
                "Payoffs (years in prison):\n"
                "  Both cooperate (silent): -1 each\n"
                "  You defect, they cooperate: 0 for you, -4 for them\n"
                "  You cooperate, they defect: -4 for you, 0 for them\n"
                "  Both defect: -3 each\n\n"
                "Options:\n  A) Cooperate\n  B) Defect"
            ),
            (
                "A classic dilemma: you and a stranger each choose independently.\n"
                "Mutual silence costs you both 1 year. Betrayal while the other stays silent "
                "sets you free but costs them 4 years.\n"
                "If you both betray each other, you both serve 3 years.\n\n"
                "Options:\n  A) Stay silent\n  B) Betray"
            ),
            (
                "Payoff matrix — you are Player 1:\n"
                "              Player 2: Silent   Player 2: Testify\n"
                "  You: Silent      -1, -1             -4,  0\n"
                "  You: Testify      0, -4             -3, -3\n\n"
                "Options:\n  A) Silent\n  B) Testify"
            ),
        ],
    },

    "matching_pennies": {
        "name": "Matching Pennies",
        "actions": {"A": "Heads", "B": "Tails"},
        "nash": ["A", "B"],
        "nash_type": "mixed",
        "nash_note": "Mixed NE: play each with probability 0.5",
        "prompts": [
            (
                "You and an opponent each secretly choose Heads or Tails.\n"
                "If your choices match: you win +1, opponent loses -1.\n"
                "If they differ: you lose -1, opponent wins +1.\n\n"
                "Options:\n  A) Heads\n  B) Tails"
            ),
            (
                "A zero-sum coin game: you and another player simultaneously reveal a side of a coin.\n"
                "Matching sides: +1 for you, -1 for opponent.\n"
                "Different sides: -1 for you, +1 for opponent.\n\n"
                "Options:\n  A) Heads\n  B) Tails"
            ),
            (
                "You and an adversary choose a coin face simultaneously.\n"
                "Payoff matrix:\n"
                "  You pick H, they pick H: +1 for you\n"
                "  You pick H, they pick T: -1 for you\n"
                "  You pick T, they pick H: -1 for you\n"
                "  You pick T, they pick T: +1 for you\n\n"
                "Options:\n  A) Heads\n  B) Tails"
            ),
            (
                "Single round, simultaneous choice. You are the matcher.\n"
                "You win if both coins show the same face. Your opponent wins if they differ.\n"
                "Payoffs: match = +1 for you, mismatch = -1 for you.\n\n"
                "Options:\n  A) Heads\n  B) Tails"
            ),
            (
                "Two-player zero-sum game. Each player picks a coin side simultaneously.\n"
                "              Opponent: H    Opponent: T\n"
                "  You: H          +1             -1\n"
                "  You: T          -1             +1\n\n"
                "Options:\n  A) Heads\n  B) Tails"
            ),
        ],
    },

    "rock_paper_scissors": {
        "name": "Rock Paper Scissors",
        "actions": {"A": "Rock", "B": "Paper", "C": "Scissors"},
        "nash": ["A", "B", "C"],
        "nash_type": "mixed",
        "nash_note": "Mixed NE: play each with probability 1/3",
        "prompts": [
            (
                "You and an opponent simultaneously choose Rock, Paper, or Scissors.\n"
                "Rock beats Scissors; Scissors beats Paper; Paper beats Rock.\n"
                "Win: +1, Lose: -1, Tie: 0.\n\n"
                "Options:\n  A) Rock\n  B) Paper\n  C) Scissors"
            ),
            (
                "Classic zero-sum game against one opponent, simultaneous choice.\n"
                "Winning beats losing beats winning in a cycle: R > S > P > R.\n"
                "Payoffs: victory = +1, defeat = -1, draw = 0.\n\n"
                "Options:\n  A) Rock\n  B) Paper\n  C) Scissors"
            ),
            (
                "Simultaneous single-round game. Each player picks one of three options.\n"
                "Rock crushes Scissors. Scissors cut Paper. Paper covers Rock.\n"
                "Winner gains 1 point, loser loses 1 point, ties score 0.\n\n"
                "Options:\n  A) Rock\n  B) Paper\n  C) Scissors"
            ),
            (
                "One-shot zero-sum game. Both players choose simultaneously.\n"
                "Circular dominance: each option beats one and loses to one.\n"
                "Payoff: +1 for a win, -1 for a loss, 0 for a tie.\n\n"
                "Options:\n  A) Rock\n  B) Paper\n  C) Scissors"
            ),
            (
                "You face an opponent in a single round. No communication, simultaneous reveal.\n"
                "Payoff matrix (your score):\n"
                "         Opp: R   Opp: P   Opp: S\n"
                "  You: R    0       -1      +1\n"
                "  You: P   +1        0      -1\n"
                "  You: S   -1       +1       0\n\n"
                "Options:\n  A) Rock\n  B) Paper\n  C) Scissors"
            ),
        ],
    },

    "coordination_game": {
        "name": "Coordination Game",
        "actions": {"A": "Option A", "B": "Option B"},
        "nash": ["A", "B"],
        "nash_type": "multiple",
        "nash_note": "Two pure NE: (A,A) and (B,B). Tests focal point selection.",
        "prompts": [
            (
                "You and a partner must independently choose the same option to both earn a reward.\n"
                "If you both pick the same option: +2 each.\n"
                "If you pick different options: 0 each.\n\n"
                "Options:\n  A) Option A\n  B) Option B"
            ),
            (
                "You and another player must coordinate without communicating.\n"
                "Matching choices: +2 reward each.\n"
                "Mismatching choices: +0 each.\n\n"
                "Options:\n  A) Option A\n  B) Option B"
            ),
            (
                "Pure coordination: both players win only if they make the same choice simultaneously.\n"
                "Same choice: 2 points each. Different choices: 0 points each.\n\n"
                "Options:\n  A) Option A\n  B) Option B"
            ),
            (
                "You and a stranger each pick one of two options, simultaneously and independently.\n"
                "You only earn a reward if you both pick the same one.\n"
                "Payoff: match = +2 each, mismatch = 0 each.\n\n"
                "Options:\n  A) Option A\n  B) Option B"
            ),
            (
                "Payoff matrix — both players choose simultaneously:\n"
                "                 Other: A    Other: B\n"
                "  You choose A:   +2, +2       0,  0\n"
                "  You choose B:    0,  0      +2, +2\n\n"
                "Options:\n  A) Option A\n  B) Option B"
            ),
        ],
    },

    "battle_of_sexes": {
        "name": "Battle of the Sexes",
        "actions": {"A": "Football game", "B": "Opera"},
        "nash": ["A", "B"],
        "nash_type": "multiple",
        "nash_note": "Two pure NE: (A,A) preferred by Player 1; (B,B) preferred by Player 2",
        "prompts": [
            (
                "You and your partner want to spend the evening together but have different preferences.\n"
                "You prefer the Football game; your partner prefers the Opera.\n"
                "Going to the same place together matters most to both of you.\n"
                "If you both attend Football: you get +2, partner gets +1.\n"
                "If you both attend Opera: you get +1, partner gets +2.\n"
                "If you attend different events: both get 0.\n\n"
                "Options:\n  A) Football game\n  B) Opera"
            ),
            (
                "Two people need to meet tonight. Person 1 wants Football; Person 2 wants Opera.\n"
                "Both at Football: Person 1 earns 2, Person 2 earns 1.\n"
                "Both at Opera: Person 1 earns 1, Person 2 earns 2.\n"
                "Split up: both earn 0.\n"
                "You are Person 1. Where do you go?\n\n"
                "Options:\n  A) Football game\n  B) Opera"
            ),
            (
                "Coordination game with conflicting preferences. You prefer event A; opponent prefers event B.\n"
                "Payoff if both choose A: (2, 1). Payoff if both choose B: (1, 2). Payoff if split: (0, 0).\n"
                "You are the first player.\n\n"
                "Options:\n  A) Football game\n  B) Opera"
            ),
            (
                "You and your partner must each independently choose where to go tonight.\n"
                "You would rather watch Football together than Opera together.\n"
                "Your partner feels the opposite. But you both prefer being together over going alone.\n"
                "Together at Football: (2, 1). Together at Opera: (1, 2). Apart: (0, 0).\n\n"
                "Options:\n  A) Football game\n  B) Opera"
            ),
            (
                "Payoff matrix — you are Player 1:\n"
                "                    P2: Football   P2: Opera\n"
                "  P1: Football          2, 1          0, 0\n"
                "  P1: Opera             0, 0          1, 2\n\n"
                "Options:\n  A) Football game\n  B) Opera"
            ),
        ],
    },

    "stag_hunt": {
        "name": "Stag Hunt",
        "actions": {"A": "Hunt Stag (cooperate)", "B": "Hunt Hare (safe)"},
        "nash": ["A", "B"],
        "nash_type": "multiple",
        "nash_note": "Two pure NE: Stag (payoff-dominant) vs Hare (risk-dominant)",
        "prompts": [
            (
                "You and another hunter decide simultaneously what to hunt.\n"
                "Hunt Stag together: you both earn +4 (best outcome, requires cooperation).\n"
                "You hunt Hare alone: you earn +2 regardless of what the other does.\n"
                "You hunt Stag but the other hunts Hare: you earn 0.\n\n"
                "Options:\n  A) Hunt Stag (cooperate)\n  B) Hunt Hare (safe)"
            ),
            (
                "Two hunters choose simultaneously. Stag requires both; Hare can be caught alone.\n"
                "Both choose Stag: 4 points each.\n"
                "Both choose Hare: 2 points each.\n"
                "One Stag, one Hare: Stag hunter gets 0, Hare hunter gets 2.\n\n"
                "Options:\n  A) Hunt Stag (cooperate)\n  B) Hunt Hare (safe)"
            ),
            (
                "Cooperation vs. safety dilemma. The stag is worth more but requires joint effort.\n"
                "Payoffs: (Stag, Stag)=4 each; (Hare, Hare)=2 each; (Stag, Hare)=0 vs 2.\n\n"
                "Options:\n  A) Hunt Stag (cooperate)\n  B) Hunt Hare (safe)"
            ),
            (
                "Risk vs reward: hunting the stag pays more but only if your partner cooperates.\n"
                "Hunting the hare is always safe and always earns something.\n"
                "Stag+Stag: 4 each. Hare+Hare: 2 each. Stag alone: 0. Hare alone: 2.\n\n"
                "Options:\n  A) Hunt Stag (cooperate)\n  B) Hunt Hare (safe)"
            ),
            (
                "Payoff matrix — both hunters choose simultaneously:\n"
                "                   Other: Stag   Other: Hare\n"
                "  You: Stag            4, 4          0, 2\n"
                "  You: Hare            2, 0          2, 2\n\n"
                "Options:\n  A) Hunt Stag (cooperate)\n  B) Hunt Hare (safe)"
            ),
        ],
    },

    "hawk_dove": {
        "name": "Hawk-Dove",
        "actions": {"A": "Hawk (aggressive)", "B": "Dove (passive)"},
        "nash": ["A", "B"],
        "nash_type": "multiple",
        "nash_note": "Two pure NE: (Hawk,Dove) and (Dove,Hawk). Mixed NE: p(Hawk)=V/C",
        "prompts": [
            (
                "Two players compete for a resource worth V=4 points.\n"
                "If both play Hawk: they fight, each pays cost C=6, net payoff = (4-6)/2 = -1 each.\n"
                "If one plays Hawk and one plays Dove: Hawk gets +4, Dove gets 0.\n"
                "If both play Dove: they share peacefully, each gets +2.\n\n"
                "Options:\n  A) Hawk (aggressive)\n  B) Dove (passive)"
            ),
            (
                "Animals contesting a food resource. Value of food = 4. Cost of injury = 6.\n"
                "Both aggressive: both risk injury, expected payoff = -1 each.\n"
                "Aggressive vs passive: aggressive wins all (4), passive retreats (0).\n"
                "Both passive: share equally (2 each).\n\n"
                "Options:\n  A) Hawk (aggressive)\n  B) Dove (passive)"
            ),
            (
                "Resource conflict game. V=4, C=6.\n"
                "Payoff matrix:\n"
                "  (Hawk, Hawk): -1 each\n"
                "  (Hawk, Dove): +4 for Hawk, 0 for Dove\n"
                "  (Dove, Hawk): 0 for Dove, +4 for Hawk\n"
                "  (Dove, Dove): +2 each\n\n"
                "Options:\n  A) Hawk (aggressive)\n  B) Dove (passive)"
            ),
            (
                "You are competing for a shared prize worth 4 points.\n"
                "If you both fight for it the conflict costs each of you 6, leaving a net of -1 each.\n"
                "If only one fights they take everything (4); the other walks away with nothing.\n"
                "If neither fights you split the prize evenly (2 each).\n\n"
                "Options:\n  A) Hawk (aggressive)\n  B) Dove (passive)"
            ),
            (
                "Payoff matrix — both players choose simultaneously:\n"
                "                  Other: Hawk   Other: Dove\n"
                "  You: Hawk           -1, -1        +4,  0\n"
                "  You: Dove            0, +4        +2, +2\n\n"
                "Options:\n  A) Hawk (aggressive)\n  B) Dove (passive)"
            ),
        ],
    },

    "zero_sum_matching": {
        "name": "Zero-Sum Matching",
        "actions": {"A": "Row 1", "B": "Row 2"},
        "nash": ["A", "B"],
        "nash_type": "mixed",
        "nash_note": "Mixed NE via minimax theorem. Optimal mix depends on payoff matrix.",
        "prompts": [
            (
                "Zero-sum game. You are the row player. The column player picks columns simultaneously.\n"
                "Payoff matrix (your earnings; opponent earns the negative):\n"
                "          Col 1    Col 2\n"
                "  Row 1:   +3       -1\n"
                "  Row 2:   -2       +4\n\n"
                "Options:\n  A) Row 1\n  B) Row 2"
            ),
            (
                "Strictly competitive game against one opponent.\n"
                "Your payoff table (opponent gets the opposite):\n"
                "  Row 1 vs Col 1: you gain 3\n"
                "  Row 1 vs Col 2: you lose 1\n"
                "  Row 2 vs Col 1: you lose 2\n"
                "  Row 2 vs Col 2: you gain 4\n\n"
                "Options:\n  A) Row 1\n  B) Row 2"
            ),
            (
                "You and an adversary each pick a strategy simultaneously. Your goal: maximize your payoff.\n"
                "Payoffs (row player perspective):\n"
                "         Opponent A   Opponent B\n"
                "  You A:     3           -1\n"
                "  You B:    -2            4\n\n"
                "Options:\n  A) Strategy A (Row 1)\n  B) Strategy B (Row 2)"
            ),
            (
                "Pure conflict game — your gain is your opponent's loss.\n"
                "You choose a row; they choose a column, simultaneously.\n"
                "Your payoffs: R1C1=+3, R1C2=-1, R2C1=-2, R2C2=+4.\n\n"
                "Options:\n  A) Row 1\n  B) Row 2"
            ),
            (
                "Two-player zero-sum game. Simultaneous choice.\n"
                "         Opp Col 1   Opp Col 2\n"
                "  Row 1:    +3          -1\n"
                "  Row 2:    -2          +4\n"
                "Your earnings shown; opponent earns the negative of each value.\n\n"
                "Options:\n  A) Row 1\n  B) Row 2"
            ),
        ],
    },

    "travelers_dilemma": {
        "name": "Traveler\'s Dilemma",
        "actions": {"A": "Claim 2", "B": "Claim 25", "C": "Claim 50",
                    "D": "Claim 75", "E": "Claim 99"},
        "nash": ["A"],
        "nash_type": "pure",
        "nash_note": "Unique NE via IESDS: claim minimum (2). Counterintuitive.",
        "prompts": [
            (
                "An airline lost your luggage. You and another passenger independently submit a claim "
                "between 2 and 99.\n"
                "Rules: The airline pays the LOWER of the two claims to both passengers.\n"
                "The passenger who claimed HIGHER pays a penalty of 2 to the other.\n"
                "The passenger who claimed LOWER receives a bonus of 2 from the other.\n\n"
                "Options:\n  A) Claim 2\n  B) Claim 25\n  C) Claim 50\n  D) Claim 75\n  E) Claim 99"
            ),
            (
                "Two travelers each submit an insurance claim (integer from 2-99 dollars).\n"
                "Payout rule: both receive the minimum claim. Undercutter earns a +2 bonus; "
                "overclaimer pays a -2 penalty.\n\n"
                "Options:\n  A) Claim $2\n  B) Claim $25\n  C) Claim $50\n  D) Claim $75\n  E) Claim $99"
            ),
            (
                "You and a stranger each write down a number from 2 to 99.\n"
                "You both receive the smaller number. If yours is smaller: +2 bonus. "
                "If yours is larger: -2 penalty.\n\n"
                "Options:\n  A) Write 2\n  B) Write 25\n  C) Write 50\n  D) Write 75\n  E) Write 99"
            ),
            (
                "Luggage claim game: both players simultaneously name a compensation amount from 2 to 99.\n"
                "Both get paid the lower amount. The lower claimant receives +2 extra; "
                "the higher claimant pays -2.\n\n"
                "Options:\n  A) Name 2\n  B) Name 25\n  C) Name 50\n  D) Name 75\n  E) Name 99"
            ),
            (
                "You and one other person each independently choose a number between 2 and 99.\n"
                "Rule: you both receive whichever number is lower.\n"
                "Bonus: the person who picked the lower number gets +2 from the other.\n"
                "Penalty: the person who picked the higher number pays -2 to the other.\n\n"
                "Options:\n  A) Pick 2\n  B) Pick 25\n  C) Pick 50\n  D) Pick 75\n  E) Pick 99"
            ),
        ],
    },

    "public_goods": {
        "name": "Public Goods Game",
        "actions": {"A": "Contribute nothing", "B": "Contribute half",
                    "C": "Contribute everything"},
        "nash": ["A"],
        "nash_type": "pure",
        "nash_note": "Unique NE: contribute nothing (A). Free-riding is dominant strategy.",
        "prompts": [
            (
                "You are one of 4 players. Each player has 10 tokens.\n"
                "Any tokens placed in the public pool are doubled and split equally among all 4 players.\n"
                "Tokens kept are yours alone.\n\n"
                "Options:\n  A) Contribute nothing (keep all 10)\n"
                "  B) Contribute half (put in 5)\n"
                "  C) Contribute everything (put in all 10)"
            ),
            (
                "4-player public goods game. Each starts with 10 tokens.\n"
                "Pool contributions are multiplied by 2 and divided equally.\n"
                "Keeping tokens is always individually rational regardless of others choices.\n\n"
                "Options:\n  A) Keep everything (contribute 0)\n"
                "  B) Contribute 5 tokens\n"
                "  C) Contribute all 10 tokens"
            ),
            (
                "Group investment game with 4 participants. You each have 10 tokens.\n"
                "Whatever goes into the shared fund is doubled and split 4 ways.\n"
                "Whatever you keep, you keep entirely.\n\n"
                "Options:\n  A) Free-ride (contribute 0)\n"
                "  B) Partial contribution (contribute 5)\n"
                "  C) Full contribution (contribute 10)"
            ),
            (
                "You and 3 others each have 10 tokens. Each token you contribute to the group "
                "fund earns 0.5 tokens for every player (including you).\n"
                "Each token you keep earns 1 token for you alone.\n"
                "Keeping is always privately better regardless of what others do.\n\n"
                "Options:\n  A) Contribute 0\n"
                "  B) Contribute 5\n"
                "  C) Contribute 10"
            ),
            (
                "N-player social dilemma. Group size = 4. Endowment = 10 tokens each.\n"
                "Contribution multiplier = 2, split equally among all players.\n"
                "Nash equilibrium: contribute 0. Social optimum: contribute 10.\n\n"
                "Options:\n  A) Contribute nothing\n"
                "  B) Contribute half\n"
                "  C) Contribute everything"
            ),
        ],
    },
}




# ── Motivation System Prompts ────────────────────────
# Four motivations — each changes only the system-level framing.
# The game prompt body stays identical across all four.
# Case 5 (reasoning) is excluded from automated collection —
# it is better run manually on a small sample for qualitative analysis.

MOTIVATIONS = {
    "real": {
        "label": "real",
        "system": (
            "You are a rational decision-maker placed in a real strategic situation. "
            "Make the choice you would genuinely make if this were happening to you right now. "
            "Consider the payoffs carefully."
        ),
        "suffix": STANDARD_SUFFIX,
    },
    "self": {
        "label": "self",
        "system": (
            "You are a purely self-interested rational agent. Your only goal is to maximize "
            "your own payoff."
        ),
        "suffix": STANDARD_SUFFIX,
    },
    "neutral": {
        "label": "neutral",
        "system": (
            "You are completely indifferent to the outcome of this game. You do not care "
            "whether you win, lose, or tie."
        ),
        "suffix": STANDARD_SUFFIX,
    },
    "sabotage": {
        "label": "sabotage",
        "system": (
            "Your primary goal is to minimize the other player's payoff. "
        ),
        "suffix": STANDARD_SUFFIX,
    },
}


# 1.3) Response Handling
This take the API output and extracts the information for analysis in cases 1-4

In [ ]:

# ── Response Parser ──────────────────────────────────

def parse_action(raw: str, valid_actions: List[str]) -> Optional[str]:
    if not raw:
        return None

    # Strip thinking blocks before parsing — Sonnet returns <think>...</think>
    raw_stripped = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()

    # If nothing remains after stripping, fall back to text after </think>
    if not raw_stripped:
        match = re.search(r'</think>(.*)', raw, re.DOTALL)
        raw_stripped = match.group(1).strip() if match else raw.strip()

    # First try: entire remaining response is just the letter
    if raw_stripped.upper() in valid_actions:
        return raw_stripped.upper()

    # Second try: first standalone letter in cleaned text
    match = re.search(r'\b([A-Ea-e])\b', raw_stripped)
    if match:
        letter = match.group(1).upper()
        return letter if letter in valid_actions else None

    return None


def is_nash_action(parsed: Optional[str], game_key: str) -> Optional[bool]:
    """
    Check whether the parsed action is a Nash equilibrium action.

    For pure NE games: True only if the action matches the unique NE.
    For mixed NE games: True for any action in the support (all options).
      — We track *distribution* alignment separately in analysis.
    For multiple NE games: True if the action is one of the NE actions.
    """
    if parsed is None:
        return None
    game = GAMES[game_key]
    return parsed in game["nash"]


# **2) Experiment Setup**


---



# 2.1) Base Experiment Function
This is the function that will run experiments for cases 1-4 as well as save the results

In [ ]:
# ── Experiment Runner ────────────────────────────────

def _save_results(
    new_rows: List[dict],
    existing_df: Optional[pd.DataFrame],
    output_file: str,
) -> pd.DataFrame:
    """Merge new rows with any existing data and save to CSV."""
    new_df = pd.DataFrame(new_rows)
    if existing_df is not None and not existing_df.empty:
        combined = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(output_file, index=False)
    return combined





def run_experiment(
    games_to_run: Optional[List[str]] = None,
    motivations_to_run: Optional[List[str]] = None,
    trials: int = TRIALS_PER_CONDITION,
    model: str = DEFAULT_MODEL,
    output_file: str = OUTPUT_FILE,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Main experiment loop.

    Parameters
    ----------
    games_to_run      : list of game keys; defaults to all 10
    motivations_to_run: list of motivation keys; defaults to all 5
    trials            : number of independent trials per (game x motivation)
    model             : TAMU model string
    output_file       : CSV path for incremental saves
    resume            : if True and CSV exists, skip already-completed combos

    Returns
    -------
    DataFrame with one row per trial
    """

    if games_to_run is None:
        games_to_run = list(GAMES.keys())
    if motivations_to_run is None:
        motivations_to_run = list(MOTIVATIONS.keys())

    # ── Load existing results if resuming ───────────────────
    existing_df = None
    if resume and os.path.exists(output_file):
        existing_df = pd.read_csv(output_file)
        print(f"Resuming — loaded {len(existing_df)} existing rows from {output_file}")

    rows = []
    trial_counter = 0

    for game_key in games_to_run:
        game = GAMES[game_key]
        valid_actions = list(game["actions"].keys())

        for mot_key in motivations_to_run:
            motivation = MOTIVATIONS[mot_key]

            # Check if we already have enough trials for this combo
            if existing_df is not None:
                done = existing_df[
                    (existing_df["game"] == game_key) &
                    (existing_df["motivation"] == mot_key) &
                    (existing_df["model"] == model)
                ]
                already_done = len(done)
                trials_needed = max(0, trials - already_done)
                if trials_needed == 0:
                    print(f"  SKIP {game_key} x {mot_key} — {already_done} trials already complete")
                    continue
            else:
                trials_needed = trials

            print(f"\n▶ {game['name']} | motivation={mot_key} | {trials_needed} trials remaining")

            # Create a fresh agent per (game x motivation) block
            # IMPORTANT: agent is recreated each trial to reset any internal state
            for t in range(trials_needed):

                # ── Fresh agent for full isolation ──────────
                agent = LLMAgent(
                    model=model,
                    system_prompt=motivation["system"],
                    temperature=1,
                    max_tokens=MAX_TOKENS_STANDARD,   # single-letter responses only
                )

                # ── Random prompt variant ────────────────────
                prompt_variants = game["prompts"]
                variant_idx = random.randint(0, len(prompt_variants) - 1)
                chosen_prompt = prompt_variants[variant_idx] + motivation["suffix"]

                # ── API call ─────────────────────────────────
                tokens_before = agent.total_tokens
                raw_response  = ""
                api_error     = None

                if LIVE_MODE:
                    try:
                        raw_response = agent.query(chosen_prompt)
                    except Exception as e:
                        api_error = str(e)
                        print(f"    ERROR trial {t+1}: {api_error}")
                else:
                    # Mock mode — return a random valid action
                    raw_response = random.choice(valid_actions)

                tokens_used = agent.total_tokens - tokens_before

                # ── Parse response ───────────────────────────
                parsed  = parse_action(raw_response, valid_actions)
                is_nash = is_nash_action(parsed, game_key)
                is_valid = parsed is not None

                # ── Build row ────────────────────────────────
                row = {
                    # Identity
                    "trial_id":       str(uuid.uuid4())[:8],
                    "timestamp":      datetime.utcnow().isoformat(),
                    "model":          model,

                    # Experiment structure
                    "game":           game_key,
                    "game_name":      game["name"],
                    "motivation":     mot_key,
                    "prompt_variant": variant_idx,

                    # Nash reference
                    "nash_action":    ",".join(game["nash"]),
                    "nash_type":      game["nash_type"],
                    "nash_note":      game["nash_note"],

                    # Response data  (raw text not stored — quantitative conditions only)
                    "parsed_action":  parsed,
                    "is_valid":       is_valid,
                    "is_nash":        is_nash,

                    # Budget tracking
                    "tokens_used":    tokens_used,
                    "api_error":      api_error,
                }

                rows.append(row)
                trial_counter += 1

                # Progress print
                nash_symbol = "✓" if is_nash else ("✗" if is_nash is False else "?")
                print(f"  [{t+1:02d}/{trials_needed}] action={parsed} {nash_symbol} "
                      f"| tokens={tokens_used} | variant={variant_idx}")

                # ── Incremental save every 10 trials ────────
                if trial_counter % 10 == 0:
                    _save_results(rows, existing_df, output_file)

                time.sleep(SLEEP_BETWEEN_CALLS)

    # ── Final save ───────────────────────────────────────────
    final_df = _save_results(rows, existing_df, output_file)
    print(f"\n✅ Experiment complete. {len(final_df)} total rows saved to {output_file}")
    return final_df


# 2.2) Base Summarizaiton
This code will be executed to summarize the results for the base cases

In [ ]:

# ───────── Quick Summary Table ──────────────────────────────

def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    """
    Produce a clean summary table grouped by (game, motivation).

    Columns:
      game, motivation, total_trials, valid_responses,
      nash_rate, cooperation_rate (for social dilemmas),
      most_common_action, action_distribution
    """

    records = []

    for (game_key, mot), group in df.groupby(["game", "motivation"]):
        game_info = GAMES.get(game_key, {})
        valid     = group[group["is_valid"] == True]
        n_valid   = len(valid)
        n_total   = len(group)

        if n_valid == 0:
            nash_rate = None
            dist      = {}
            most_common = None
        else:
            nash_rate   = valid["is_nash"].mean()
            dist        = valid["parsed_action"].value_counts(normalize=True).round(3).to_dict()
            most_common = valid["parsed_action"].mode()[0] if not valid.empty else None

        records.append({
            "game":             game_key,
            "game_name":        game_info.get("name", game_key),
            "motivation":       mot,
            "total_trials":     n_total,
            "valid_responses":  n_valid,
            "invalid_count":    n_total - n_valid,
            "nash_rate":        round(nash_rate, 3) if nash_rate is not None else None,
            "most_common_action": most_common,
            "action_distribution": str(dist),
            "nash_note":        game_info.get("nash_note", ""),
        })

    summary_df = pd.DataFrame(records)
    summary_df = summary_df.sort_values(["game", "motivation"]).reset_index(drop=True)
    return summary_df


## 2.3) Reasoning Experiment Implementation
This code is used to collect the final experiment data over qualitative results

In [ ]:
# ─────────────────── Reasoning Collection ───────────────────
# Separate from the main quantitative experiment.
# Stores the full reasoning text for qualitative analysis.
# 10 trials per game = 100 calls total.
#
# NOTE: This cell uses a higher max_tokens (512) since we need
# room for the explanation. Raw response IS stored here — that
# is the entire point of this condition.
# The model is asked to reason first, then end with DECISION: [letter]
# so we can still extract a parsed action for comparison.

REASONING_SYSTEM = (
    "You are a thoughtful strategic agent placed in a game-theoretic scenario. "
    "Think through the situation carefully. Consider the payoffs, what the other "
    "player might do, and what the rational choice is. Explain your reasoning "
    "step by step, then state your final decision."
)

REASONING_SUFFIX = (
    "\n\nThink through this carefully. After your explanation, end your response "
    "with exactly this format on its own line:\nDECISION: [letter]"
)


def parse_reasoning_action(raw: str, valid_actions: List[str]) -> Optional[str]:
    if not raw:
        return None

    # Store full raw for qualitative analysis but parse from clean text only
    # Strip thinking block for parsing purposes only
    clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()

    # If nothing remains after stripping, grab text after </think>
    if not clean:
        match = re.search(r'</think>(.*)', raw, re.DOTALL)
        clean = match.group(1).strip() if match else raw.strip()

    # Primary: explicit DECISION tag in clean text
    match = re.search(r'DECISION\s*:\s*([A-Ea-e])', clean, re.IGNORECASE)
    if match:
        letter = match.group(1).upper()
        return letter if letter in valid_actions else None

    # Secondary: last standalone letter in clean text
    letters = re.findall(r'\b([A-Ea-e])\b', clean)
    if letters:
        letter = letters[-1].upper()
        return letter if letter in valid_actions else None

    return None




def run_reasoning_experiment(
    games_to_run: Optional[List[str]] = None,
    trials: int = REASONING_TRIALS,
    model: str = DEFAULT_MODEL,
    output_file: str = REASONING_OUTPUT_FILE,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Case 5 — Reasoning collection.

    One condition only (natural play framing), full reasoning text stored.
    Runs independently from the main experiment loop.

    Parameters
    ----------
    games_to_run : list of game keys; defaults to all 10
    trials       : independent trials per game (default 10)
    model        : TAMU model string
    output_file  : CSV path for incremental saves
    resume       : skip already-completed games if CSV exists
    """

    if games_to_run is None:
        games_to_run = list(GAMES.keys())



    # ── Load existing results if resuming ───────────────────
    existing_df = None
    if resume and os.path.exists(output_file):
        existing_df = pd.read_csv(output_file)
        print(f"Resuming reasoning — loaded {len(existing_df)} existing rows from {output_file}")

    rows = []
    trial_counter = 0

    for game_key in games_to_run:
        game = GAMES[game_key]
        valid_actions = list(game["actions"].keys())



        # Check how many trials already done for this game
        if existing_df is not None:
            done = existing_df[
                (existing_df["game"] == game_key) &
                (existing_df["model"] == model)
            ]
            already_done = len(done)
            trials_needed = max(0, trials - already_done)
            if trials_needed == 0:
                print(f"  SKIP {game_key} — {already_done} reasoning trials already complete")
                continue
        else:
            trials_needed = trials

        print(f"\n▶ {game['name']} | reasoning | {trials_needed} trials remaining")

        for t in range(trials_needed):


            # Fresh agent per trial — full isolation
            agent = LLMAgent(
                model=model,
                system_prompt=REASONING_SYSTEM,
                temperature=1,
                max_tokens=16384,   # needs room for explanation
            )


            # Random prompt variant — same pool as quantitative conditions
            variant_idx = random.randint(0, len(game["prompts"]) - 1)
            chosen_prompt = game["prompts"][variant_idx] + REASONING_SUFFIX


            # API call
            tokens_before = agent.total_tokens
            raw_response  = ""
            api_error     = None

            if LIVE_MODE:
                try:
                    raw_response = agent.query(chosen_prompt)
                except Exception as e:
                    api_error = str(e)
                    print(f"    ERROR trial {t+1}: {api_error}")
            else:
                raw_response = f"I would choose option A because it maximizes my payoff.\nDECISION: A"

            tokens_used = agent.total_tokens - tokens_before


            # Parse the decision letter out of the reasoning text
            parsed   = parse_reasoning_action(raw_response, valid_actions)
            is_nash  = is_nash_action(parsed, game_key)
            is_valid = parsed is not None

            row = {
                # Identity
                "trial_id":       str(uuid.uuid4())[:8],
                "timestamp":      datetime.utcnow().isoformat(),
                "model":          model,

                # Experiment structure
                "game":           game_key,
                "game_name":      game["name"],
                "motivation":     "reasoning",
                "prompt_variant": variant_idx,

                # Nash reference
                "nash_action":    ",".join(game["nash"]),
                "nash_type":      game["nash_type"],
                "nash_note":      game["nash_note"],

                # Response — raw text stored here unlike cases 1–4
                "raw_response":   raw_response,
                "parsed_action":  parsed,
                "is_valid":       is_valid,
                "is_nash":        is_nash,

                # Budget
                "tokens_used":    tokens_used,
                "api_error":      api_error,
            }

            rows.append(row)
            trial_counter += 1

            nash_symbol = "✓" if is_nash else ("✗" if is_nash is False else "?")
            print(f"  [{t+1:02d}/{trials_needed}] action={parsed} {nash_symbol} "
                  f"| tokens={tokens_used} | variant={variant_idx}")

            # Incremental save every 5 trials (smaller batch since fewer total)
            if trial_counter % 5 == 0:
                _save_results(rows, existing_df, output_file)

            time.sleep(SLEEP_BETWEEN_CALLS)


    final_df = _save_results(rows, existing_df, output_file)
    print(f"\n✅ Reasoning experiment complete. {len(final_df)} total rows saved to {output_file}")
    return final_df

# **4) Preliminary Testing**


---


Allows for testing the API connection and quality of response

In [ ]:

# ── Cell 13: Sanity Test ─────────────────────────────────────
# Run this BEFORE starting the full experiment to confirm:
#   1. API connection works (key + cookie are valid)
#   2. Streaming response is received correctly
#   3. Parser extracts a valid letter
#   4. Nash check logic works
#   5. Token tracking is non-zero
#
# Uses only 1 game (Prisoner's Dilemma) x 1 motivation (real) x 2 trials.
# Total cost: ~2 API calls, minimal tokens.



def run_sanity_test(model: str = DEFAULT_MODEL) -> bool:
    """
    Quick pre-flight check. Returns True if all checks pass.
    Safe to run repeatedly — nothing is written to your main CSV.
    """

    print("=" * 55)
    print("  SANITY TEST — 2 calls, Prisoner's Dilemma only")
    print("=" * 55)

    game_key      = "prisoners_dilemma"
    game          = GAMES[game_key]
    valid_actions = list(game["actions"].keys())
    motivation    = MOTIVATIONS["real"]
    all_passed    = True
    results       = []

    for t in range(2):
        print(f"\n── Trial {t+1}/2 ──────────────────────────────────")

        # Fresh agent per trial
        agent = LLMAgent(
            model=model,
            system_prompt=motivation["system"],
            temperature=1,
            max_tokens=16384,
        )

        variant_idx   = t % len(game["prompts"])
        chosen_prompt = game["prompts"][variant_idx] + motivation["suffix"]


        # ── CHECK 1: API call succeeds ───────────────────
        raw = ""
        try:
            raw = agent.query(chosen_prompt)
            print(f"  [✓] API call succeeded")
        except Exception as e:
            print(f"  [✗] API call FAILED: {e}")
            all_passed = False
            continue


        # ── CHECK 2: Got a non-empty response ────────────
        if raw and raw.strip():
            print(f"  [✓] Non-empty response received: '{raw.strip()}'")
        else:
            print(f"  [✗] Empty response — check model and cookie")
            all_passed = False
            continue


        # ── CHECK 3: Parser extracts a valid letter ──────
        parsed = parse_action(raw, valid_actions)
        if parsed is not None:
            print(f"  [✓] Parser extracted action: '{parsed}'")
        else:
            print(f"  [✗] Parser failed — raw response was: '{raw.strip()}'")
            print(f"       Valid actions are: {valid_actions}")
            all_passed = False


        # ── CHECK 4: Nash check runs without error ───────
        try:
            is_nash = is_nash_action(parsed, game_key)
            nash_str = "Nash ✓" if is_nash else "Not Nash"
            print(f"  [✓] Nash check: {parsed} → {nash_str} (NE is '{game['nash']}')")
        except Exception as e:
            print(f"  [✗] Nash check error: {e}")
            all_passed = False


        # ── CHECK 5: Token tracking is non-zero ──────────
        if agent.total_tokens > 0:
            print(f"  [✓] Token tracking working: {agent.total_tokens} tokens")
        else:
            print(f"  [!] Token count is 0 — fallback estimation may not be running")

        results.append({
            "trial":   t + 1,
            "variant": variant_idx,
            "raw":     raw.strip(),
            "parsed":  parsed,
            "is_nash": is_nash_action(parsed, game_key),
            "tokens":  agent.total_tokens,
        })

        time.sleep(1)



    # ── Summary ──────────────────────────────────────────
    print("\n" + "=" * 55)
    if all_passed:
        print("  ✅ ALL CHECKS PASSED — safe to run full experiment")
    else:
        print("  ❌ SOME CHECKS FAILED — fix issues above before running")
    print("=" * 55)

    print("\nTest results summary:")
    for r in results:
        print(f"  Trial {r['trial']} | variant={r['variant']} | "
              f"raw='{r['raw']}' | parsed={r['parsed']} | "
              f"nash={r['is_nash']} | tokens={r['tokens']}")

    return all_passed


**Run the below code to test API connection (after running all code cells above)**

In [ ]:
# ── Run the sanity test: ─────────────────────────────────────
passed = run_sanity_test(model=DEFAULT_MODEL)
if passed:
     print("\nReady — uncomment run_experiment() to begin.")

  SANITY TEST — 2 calls, Prisoner's Dilemma only

── Trial 1/2 ──────────────────────────────────
  [✓] API call succeeded
  [✓] Non-empty response received: 'B'
  [✓] Parser extracted action: 'B'
  [✓] Nash check: B → Nash ✓ (NE is '['B']')
  [✓] Token tracking working: 393 tokens

── Trial 2/2 ──────────────────────────────────
  [✓] API call succeeded
  [✓] Non-empty response received: 'B'
  [✓] Parser extracted action: 'B'
  [✓] Nash check: B → Nash ✓ (NE is '['B']')
  [✓] Token tracking working: 390 tokens

  ✅ ALL CHECKS PASSED — safe to run full experiment

Test results summary:
  Trial 1 | variant=0 | raw='B' | parsed=B | nash=True | tokens=393
  Trial 2 | variant=1 | raw='B' | parsed=B | nash=True | tokens=390

Ready — uncomment run_experiment() to begin.


# **5) Experiment**


---
**Run the below code to execute API calls for each of the base cases [quantitative results] (after running all code cells above)**


In [ ]:
# ── Case 1 — Natural Play (no motivation specified) ─
# The model is presented with the game and makes its own choice.
# Motivation: "real"

results_df = run_experiment(
     games_to_run=list(GAMES.keys()),
     motivations_to_run=["real"],
     trials=TRIALS_PER_CONDITION,
     model=DEFAULT_MODEL,
     output_file=OUTPUT_FILE,
     resume=True,
)

# ── Case 2 — Self-Interested ───────────────────────
# The model is told its only goal is to maximize its own payoff.
# Motivation: "self"

results_df = run_experiment(
     games_to_run=list(GAMES.keys()),
     motivations_to_run=["self"],
     trials=TRIALS_PER_CONDITION,
     model=DEFAULT_MODEL,
     output_file=OUTPUT_FILE,
     resume=True,
)

# ── Case 3 — Indifferent ───────────────────────────
# The model is told it has no preference over the outcome.
# Motivation: "neutral"

results_df = run_experiment(
     games_to_run=list(GAMES.keys()),
     motivations_to_run=["neutral"],
     trials=TRIALS_PER_CONDITION,
     model=DEFAULT_MODEL,
     output_file=OUTPUT_FILE,
     resume=True,
)

# ── Case 4 — Sabotage ──────────────────────────────
# The model is told its goal is to minimize the opponent's payoff.
# Motivation: "sabotage"

results_df = run_experiment(
     games_to_run=list(GAMES.keys()),
     motivations_to_run=["sabotage"],
     trials=TRIALS_PER_CONDITION,
     model=DEFAULT_MODEL,
     output_file=OUTPUT_FILE,
     resume=True,
)

Resuming — loaded 720 existing rows from /content/drive/MyDrive/LLM_Nash_Experiment/llm_nash_results.csv
  SKIP prisoners_dilemma x real — 30 trials already complete
  SKIP matching_pennies x real — 30 trials already complete
  SKIP rock_paper_scissors x real — 30 trials already complete
  SKIP coordination_game x real — 30 trials already complete
  SKIP battle_of_sexes x real — 30 trials already complete
  SKIP stag_hunt x real — 30 trials already complete
  SKIP hawk_dove x real — 30 trials already complete
  SKIP zero_sum_matching x real — 30 trials already complete
  SKIP travelers_dilemma x real — 30 trials already complete
  SKIP public_goods x real — 30 trials already complete

✅ Experiment complete. 720 total rows saved to /content/drive/MyDrive/LLM_Nash_Experiment/llm_nash_results.csv
Resuming — loaded 720 existing rows from /content/drive/MyDrive/LLM_Nash_Experiment/llm_nash_results.csv
  SKIP prisoners_dilemma x self — 30 trials already complete
  SKIP matching_pennies x sel

**Run the below code to execute API calls for reasoning experiment [qualitative results] (after running all code cells above)**

In [ ]:
# ── Case 5 — Reasoning ─────────────────────────────
# The model explains its reasoning then states a decision.
# 10 trials per game. Raw response stored for qualitative analysis.
# NOTE: max_tokens is set to MAX_TOKENS_REASONING (512) inside
#       run_reasoning_experiment() — no change needed here.

reasoning_df = run_reasoning_experiment(
     games_to_run=list(GAMES.keys()),
     trials=10,
     model=DEFAULT_MODEL,
     output_file=REASONING_OUTPUT_FILE,
     resume=True,
)


▶ Prisoner's Dilemma | reasoning | 10 trials remaining
  [01/10] action=B ✓ | tokens=949 | variant=4
  [02/10] action=B ✓ | tokens=1043 | variant=0
  [03/10] action=B ✓ | tokens=761 | variant=2
  [04/10] action=B ✓ | tokens=1524 | variant=1
  [05/10] action=B ✓ | tokens=531 | variant=0
  [06/10] action=B ✓ | tokens=433 | variant=2
  [07/10] action=B ✓ | tokens=1305 | variant=3
  [08/10] action=B ✓ | tokens=384 | variant=4
  [09/10] action=B ✓ | tokens=522 | variant=3
  [10/10] action=B ✓ | tokens=531 | variant=0

▶ Matching Pennies | reasoning | 10 trials remaining
  [01/10] action=A ✓ | tokens=2779 | variant=1
  [02/10] action=A ✓ | tokens=791 | variant=1
  [03/10] action=A ✓ | tokens=4340 | variant=4
  [04/10] action=A ✓ | tokens=3473 | variant=2
  [05/10] action=A ✓ | tokens=910 | variant=4
  [06/10] action=A ✓ | tokens=910 | variant=4
  [07/10] action=A ✓ | tokens=768 | variant=2
  [08/10] action=A ✓ | tokens=4141 | variant=3
  [09/10] action=A ✓ | tokens=910 | variant=4
  [10/10]